In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Building a Texr Cleaning Pipeline

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download if first time
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Cleans text data:
    - lowercase
    - remove URLs
    - remove emojis
    - remove symbols
    - remove stopwords
    - lemmatize or stem
    """

    # Convert to lowercase
    data = dataset.lower()

    # Remove URLs
    data = re.sub(r"http\S+|www\S+|https\S+", '', data)

    # Remove emojis
    data = re.sub(r'[^\x00-\x7F]+', '', data)

    # Remove unwanted characters
    data = re.sub(r'[^a-zA-Z\s]', '', data)

    # Tokenize
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize or Stem
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]

    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)


# Text Classification using Machine Learning Models

### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [ ]:
#importing libraries

In [ ]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Download NLTK resources (run once)
nltk.download('stopwords')
nltk.download('wordnet')


In [ ]:
#loading dataset
import os
print(os.listdir())

In [ ]:
print(os.listdir('/content/drive/MyDrive/Sem6AI/Week8'))

In [ ]:
#loading the dataset
df= pd.read_csv('/content/drive/MyDrive/Sem6AI/Week8/trum_tweet_sentiment_analysis.csv')

In [ ]:
# Show first 5 rows
print(df.head())

# Check required columns
print("\nColumns:", df.columns)

In [ ]:
df = df[['text', 'Sentiment']]
df.rename(columns={'Sentiment':'label'}, inplace=True)

In [ ]:
# Keep only needed columns
df = df[['text', 'label']]

# Remove missing values
df.dropna(inplace=True)

print("\nDataset Shape:", df.shape)

In [ ]:
#text cleaning function

In [ ]:

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Cleans text data:
    - lowercase
    - remove URLs
    - remove mentions
    - remove punctuation
    - remove special chars
    - remove stopwords
    - tokenize
    - lemmatize or stem
    """

    # Convert to lowercase
    data = dataset.lower()

    # Remove URLs
    data = re.sub(r"http\S+|www\S+|https\S+", '', data)

    # Remove mentions (@username)
    data = re.sub(r'@\w+', '', data)

    # Remove hashtags symbol only (#trump -> trump)
    data = re.sub(r'#', '', data)

    # Remove punctuation / special characters / numbers
    data = re.sub(r'[^a-zA-Z\s]', '', data)

    # Tokenization
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization or Stemming
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]

    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

In [ ]:
#applying cleaning

In [ ]:
df['clean_text'] = df['text'].apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))

print("\nCleaned Text Sample:")
print(df[['text', 'clean_text']].head())

In [ ]:
#defining features and labels

In [ ]:
X = df['clean_text']
y = df['label']

In [ ]:
#training and testing split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Size:", X_train.shape[0])
print("Testing Size:", X_test.shape[0])

In [ ]:
#TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF Shape:", X_train_tfidf.shape)

In [ ]:
#training Logisitic Regression Model
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

In [ ]:
#predicting
y_pred = model.predict(X_test_tfidf)

In [ ]:
#evaluating
print("\nAccuracy Score:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))